*The `PM2.5_2017.csv` dataset exceeds GitHub's 100 MB file size limit, so it is stored as a GitHub Release asset instead. The Census dataset `dp03.csv` is stored directly in the repository. The code below automatically downloads either dataset if it is missing locally, allowing the notebook to run without manually downloading the files.*

In [178]:
from pathlib import Path
from urllib.request import urlretrieve
import pandas as pd
import numpy as np


#Location where the datasets should be stored
pm25_path = Path("Datasets/PM2.5_2017.csv")
census_path = Path("Datasets/dp03.csv")

#GitHub download URLs
pm25_url = "https://github.com/alcndram/5574-final/releases/download/v1.0/PM2.5_2017.csv"
census_url = "https://github.com/alcndram/5574-final/releases/download/v1.0/dp03.csv"

#Automatically downloads datasets if missing 

if not pm25_path.exists():
    pm25_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(pm25_url, pm25_path)

if not census_path.exists():
    census_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(census_url, census_path)

#Loading datasets
pm25 = pd.read_csv(pm25_path, low_memory=False)
census = pd.read_csv(census_path, skiprows=[1], low_memory=False)


# Section 1: Importing and Cleaning Data

We chose PM2.5 because it gives us one clear way to measure and compare air quality across counties. Focusing on one pollutant keeps the analysis simple and avoids trying to combine pollutants that may be measured differently.

In [179]:
pm25 = pd.read_csv('Datasets/PM2.5_2017.csv', low_memory=False) #set low_memory to false to avoid dtype warning

census = pd.read_csv(
    'Datasets/dp03.csv',
    skiprows=[1],
    low_memory=False    #set low_memory to false to avoid dtype warning
)


In [180]:
# cleaning the PM2.5 dataset

#Keeping important columns for analysis

pm25 = pm25[[
    'State Code',
    'County Code',
    'Site Num',
    'Date Local',
    'POC',
    'Pollutant Standard',
    'Arithmetic Mean',
    'State Name',
    'County Name'
]].copy()

#Dataset contains several kinds of daily summaries
#Interested in daily PM2.5 concentrations, keeping records associated with the 24-hour PM2.5 standard:

pm25 = pm25[pm25['Pollutant Standard'] == 'PM25 24-hour 2012'].copy()

#Converting the date
pm25['Date Local'] = pd.to_datetime(pm25['Date Local'], format='%Y-%m-%d')

We created a county FIPS code to use as the common key between the EPA and Census datasets. We chose FIPS rather than county names because county names can be formatted differently across datasets or repeated in different states.

Combining the state and county codes creates a unique identifier for each county.

In [181]:
pm25['FIPS'] = (pm25['State Code'].astype(str).str.zfill(2) 
+ 
pm25['County Code'].astype(str).str.zfill(3))

Now, we take into consideration that there can be more than one EPA record for the same monitoring site on the same day. 

In [182]:
pm25_daily = pm25.groupby(['FIPS', 
                           'State Name',
                           'County Name',
                           'Site Num',
                           'Date Local'])['Arithmetic Mean'].mean().reset_index()

#Renaming the column to be more descriptive
pm25_daily.rename(columns={'Arithmetic Mean': 'Daily PM2.5 Concentration'}, inplace=True)


In [183]:
pm25_daily.head()

,FIPS,State Name,County Name,Site Num,Date Local,Daily PM2.5 Concentration
0,01003,Alabama,Baldwin,10,2017-01-01,5.0
1,01003,Alabama,Baldwin,10,2017-01-04,5.3
2,01003,Alabama,Baldwin,10,2017-01-10,4.5
3,01003,Alabama,Baldwin,10,2017-01-13,7.2
4,01003,Alabama,Baldwin,10,2017-01-16,3.9


Next, we are taking those daily monitor measurements and summarizing them by county. 

In [184]:
county_pm25 = pm25_daily.groupby(['FIPS', 
                                  'State Name', 
                                  'County Name'])['Daily PM2.5 Concentration'].mean().reset_index()

county_pm25.rename(columns={'Daily PM2.5 Concentration': 'Average PM2.5 Concentration'}, inplace=True)


Next, cleaning up the Census dataset.

In [185]:
census = census[['GEO_ID', 
                 'NAME', 
                 'DP03_0062E', 
                 'DP03_0009PE',
                 'DP03_0128PE']].copy()


In [186]:
census.rename(columns={
    'NAME': 'Census_Name',
    'DP03_0062E': 'Median_Household_Income',
    'DP03_0009PE': 'Unemployment_Rate',
    'DP03_0128PE': 'Poverty_Rate'
}, inplace=True)




In [187]:
census['FIPS'] = (
    census['GEO_ID']
    .astype(str)
    .str[-5:]
)

In [188]:
census.head()

,GEO_ID,Census_Name,Median_Household_Income,Unemployment_Rate,Poverty_Rate,FIPS
0,0500000US01001,"Autauga County, Alabama",55317,5.2,13.7,01001
1,0500000US01003,"Baldwin County, Alabama",52562,5.5,11.8,01003
2,0500000US01005,"Barbour County, Alabama",33368,12.4,27.2,01005
3,0500000US01007,"Bibb County, Alabama",43404,8.2,15.2,01007
4,0500000US01009,"Blount County, Alabama",47412,4.9,15.6,01009


In [189]:
#converting the columns to numeric values

census['Median_Household_Income'] = pd.to_numeric(census['Median_Household_Income'], errors='coerce')
census['Unemployment_Rate'] = pd.to_numeric(census['Unemployment_Rate'], errors='coerce')
census['Poverty_Rate'] = pd.to_numeric(census['Poverty_Rate'], errors='coerce')

In [190]:
census.head()

,GEO_ID,Census_Name,Median_Household_Income,Unemployment_Rate,Poverty_Rate,FIPS
0,0500000US01001,"Autauga County, Alabama",55317,5.2,13.7,01001
1,0500000US01003,"Baldwin County, Alabama",52562,5.5,11.8,01003
2,0500000US01005,"Barbour County, Alabama",33368,12.4,27.2,01005
3,0500000US01007,"Bibb County, Alabama",43404,8.2,15.2,01007
4,0500000US01009,"Blount County, Alabama",47412,4.9,15.6,01009


Merge datasets.

We did an inner join to keep only counties that have both PM2.5 and census data. Using a left join would keep all counties in PM2.5 dataset and fill missing values for census data with NaN.

In [191]:
df = county_pm25.merge(census, on='FIPS', how='inner')


In [192]:
df.isnull().sum() #checking for missing values

df = df.drop(columns=['GEO_ID']) #dropping unnecessary column

df.head()

,FIPS,State Name,County Name,Average PM2.5 Concentration,Census_Name,Median_Household_Income,Unemployment_Rate,Poverty_Rate
0,01003,Alabama,Baldwin,7.391818,"Baldwin County, Alabama",52562,5.5,11.8
1,01027,Alabama,Clay,7.906780,"Clay County, Alabama",37287,6.4,19.1
2,01033,Alabama,Colbert,7.382353,"Colbert County, Alabama",45477,6.9,16.8
3,01049,Alabama,DeKalb,8.006087,"DeKalb County, Alabama",38842,5.1,21.5
4,01055,Alabama,Etowah,8.849558,"Etowah County, Alabama",42064,7.6,17.9
